In [ ]:
import pandas as pd

# CSV-Datei einlesen
# Quelle: https://www.landesdatenbank.nrw.de/
df_35a = pd.read_csv("../data/raw/2024/22811-01i.csv", sep=";", encoding="latin1", skiprows=3)

# Überblick:
print(df_35a.head())
print(df_35a.info())

In [ ]:
#relevante Spalten filtern
df = df_35a[['Unnamed: 1','Unnamed: 2']]
df = df.dropna(axis=1, how="all")
df = df.dropna(axis=0, how="all")
print(df)

In [ ]:
# Spalten umbenennen
df = df.rename(columns={"Unnamed: 1": "Name", "Unnamed: 2": "Mindestsicherungsquote"})

# nach Kreisen und kreisfreien Städten filtern und Strings anpassen
df = df[df["Name"].str.contains("Kreis|krfr. Stadt|Städteregion", case=False,na=False)]
df["Name"] = df["Name"].str.strip()

df["Mindestsicherungsquote"] = (
    df["Mindestsicherungsquote"]
    .astype(str)                
    .str.replace(",", ".", regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors="coerce")
)


print(df)

In [ ]:
# Zeile löschen und Name anpassen
df = df[df["Name"] != "Aachen, krfr. Stadt (ab 21.10.2009)"]
df = df[df["Name"] != "Essen, krfr. Stadt"]

# fehlende Werte anzeigen
print(df[df["Mindestsicherungsquote"].isna()])

In [ ]:
from ipynb.fs.defs.functions import clean_and_sort, validate_df
# Tabelle nach Kreis/Stadt sortieren
df = clean_and_sort(df, "Mindestsicherungsquote")
validate_df(
    df,
    not_null=["Mindestsicherungsquote"],
    non_negative=["Mindestsicherungsquote"],
    numeric=["Mindestsicherungsquote"],
    bounds={"Mindestsicherungsquote": (0,100)},
    key_cols=["Name"],
)

In [ ]:
# Übersicht
df["Mindestsicherungsquote"].describe()

In [ ]:
# speichern
df.to_csv("../data/processed/mindestsicherungsquote_2024.csv", index=False)